<h2>Fetching the datasets from kaggle</h2>

In [11]:
# Import necessary libraries
import pandas as pd
import numpy as np
import kagglehub
from kaggle.api.kaggle_api_extended import KaggleApi
from pathlib import Path

In [12]:
from dotenv import load_dotenv
import os

load_dotenv()
kaggle_username = os.getenv('KAGGLE_USERNAME')
kaggle_key = os.getenv('KAGGLE_KEY')

api = KaggleApi()
api.authenticate()

In [13]:
# Step 1: Define Kaggle datasets to fetch
datasets = {
    "dataset1": "zgrcemta/world-gdpgdp-gdp-per-capita-and-annual-growths",
    "dataset2": "unsdsn/world-happiness",
    "dataset3": "pantanjali/unemployment-dataset"
}

# Create a data folder to store datasets
data_path = Path("data")
data_path.mkdir(exist_ok=True)

# Step 2: Download datasets
for name, dataset in datasets.items():
    api.dataset_download_files(dataset, path=str(data_path / name), unzip=True)

# Step 3: Delete gpd.csv, gdp_ppp.csv and gdp_growth file from dataset1
gdp_path = data_path / "dataset1"
gdp_path.joinpath("gdp.csv").unlink()
gdp_path.joinpath("gdp_ppp.csv").unlink()
gdp_path.joinpath("gdp_growth.csv").unlink()
gdp_path.joinpath("gdp_ppp_per_capita.csv").unlink()


Dataset URL: https://www.kaggle.com/datasets/zgrcemta/world-gdpgdp-gdp-per-capita-and-annual-growths
Dataset URL: https://www.kaggle.com/datasets/unsdsn/world-happiness
Dataset URL: https://www.kaggle.com/datasets/pantanjali/unemployment-dataset


<h2>Load the data</h2>

In [14]:
#load the happiness_data.csv
happiness_data = pd.read_csv('data/happiness_data.csv')

# Step 3: Load datasets
datasets_loaded = {}
for name in datasets.keys():
    dataset_dir = data_path / name
    csv_files = list(dataset_dir.glob("*.csv"))
    datasets_loaded[name] = {csv_file.stem: pd.read_csv(csv_file) for csv_file in csv_files}


Combine the yearly happiness datasets from 2015 to 2019 into a single dataset (_no need to run if happiness_data.csv is already available_)

In [15]:
#remove all columns execpt from country and happiness score
datasets_loaded['dataset2']['2015'] = datasets_loaded['dataset2']['2015'][['Country', 'Happiness Score']]
datasets_loaded['dataset2']['2016'] = datasets_loaded['dataset2']['2016'][['Country', 'Happiness Score']]
datasets_loaded['dataset2']['2017'] = datasets_loaded['dataset2']['2017'][['Country', 'Happiness.Score']]
datasets_loaded['dataset2']['2018'] = datasets_loaded['dataset2']['2018'][['Country or region', 'Score']]
datasets_loaded['dataset2']['2019'] = datasets_loaded['dataset2']['2019'][['Country or region', 'Score']]

#rename the columns to be the same
datasets_loaded['dataset2']['2015'].rename(columns={'Happiness Score': 'Happiness_Score_2015'}, inplace=True)
datasets_loaded['dataset2']['2016'].rename(columns={'Happiness Score': 'Happiness_Score_2016'}, inplace=True)
datasets_loaded['dataset2']['2017'].rename(columns={'Happiness.Score': 'Happiness_Score_2017'}, inplace=True)
datasets_loaded['dataset2']['2018'].rename(columns={'Country or region': 'Country', 'Score': 'Happiness_Score_2018'}, inplace=True)
datasets_loaded['dataset2']['2019'].rename(columns={'Country or region': 'Country', 'Score': 'Happiness_Score_2019'}, inplace=True)

#merge the above datasets on the country column
happiness_data = pd.merge(datasets_loaded['dataset2']['2015'], datasets_loaded['dataset2']['2016'], on='Country', how='inner')
happiness_data = pd.merge(happiness_data, datasets_loaded['dataset2']['2017'], left_on='Country', right_on='Country', how='inner')
happiness_data = pd.merge(happiness_data, datasets_loaded['dataset2']['2018'], left_on='Country', right_on='Country', how='inner')
happiness_data = pd.merge(happiness_data, datasets_loaded['dataset2']['2019'], left_on='Country', right_on='Country', how='inner')

#save the final dataset
happiness_data.to_csv('data/happiness_data.csv', index=False)

#print the final datasets
print(happiness_data)

         Country  Happiness_Score_2015  Happiness_Score_2016  \
0    Switzerland                 7.587                 7.509   
1        Iceland                 7.561                 7.501   
2        Denmark                 7.527                 7.526   
3         Norway                 7.522                 7.498   
4         Canada                 7.427                 7.404   
..           ...                   ...                   ...   
136       Rwanda                 3.465                 3.515   
137        Benin                 3.340                 3.484   
138        Syria                 3.006                 3.069   
139      Burundi                 2.905                 2.905   
140         Togo                 2.839                 3.303   

     Happiness_Score_2017  Happiness_Score_2018  Happiness_Score_2019  
0                   7.494                 7.487                 7.480  
1                   7.504                 7.495                 7.494  
2              

In [16]:
employment_data = pd.read_csv('data/dataset3/Unemployment analysis.csv')

#keep only the columns we need and rename to the universal index: Country
employment_data = employment_data[['Country Name', '2015', '2016', '2017', '2018', '2019']]
employment_data.rename(columns={'Country Name': 'Country'}, inplace=True)
#rename all yearly columns to unemployment_rate_x
employment_data.rename(columns={'2015': 'Unemployment_Rate_2015'}, inplace=True)
employment_data.rename(columns={'2016': 'Unemployment_Rate_2016'}, inplace=True)
employment_data.rename(columns={'2017': 'Unemployment_Rate_2017'}, inplace=True)
employment_data.rename(columns={'2018': 'Unemployment_Rate_2018'}, inplace=True)
employment_data.rename(columns={'2019': 'Unemployment_Rate_2019'}, inplace=True)

#print a sample dataset
print(employment_data.head())

#save the final dataset
employment_data.to_csv('data/employment_data.csv', index=False)

                       Country  Unemployment_Rate_2015  \
0  Africa Eastern and Southern                    6.49   
1                  Afghanistan                   11.13   
2   Africa Western and Central                    4.63   
3                       Angola                    7.39   
4                      Albania                   17.19   

   Unemployment_Rate_2016  Unemployment_Rate_2017  Unemployment_Rate_2018  \
0                    6.61                    6.71                    6.73   
1                   11.16                   11.18                   11.15   
2                    5.57                    6.02                    6.04   
3                    7.41                    7.41                    7.42   
4                   15.42                   13.62                   12.30   

   Unemployment_Rate_2019  
0                    6.91  
1                   11.22  
2                    6.06  
3                    7.42  
4                   11.47  


In [ ]:
#do the same as above for the gdp dataset

gdp_data = pd.read_csv('data/dataset1/gdp_per_capita.csv')

#keep only the columns we need and rename to the universal index: Country
gdp_data = gdp_data[['Country Name', '2015', '2016', '2017', '2018', '2019']]
gdp_data.rename(columns={'Country Name': 'Country'}, inplace=True)

#rename all yearly columns to gdp_per_capita_x
gdp_data.rename(columns={'2015': 'GDP_Per_Capita_2015'}, inplace=True)
gdp_data.rename(columns={'2016': 'GDP_Per_Capita_2016'}, inplace=True)
gdp_data.rename(columns={'2017': 'GDP_Per_Capita_2017'}, inplace=True)
gdp_data.rename(columns={'2018': 'GDP_Per_Capita_2018'}, inplace=True)
gdp_data.rename(columns={'2019': 'GDP_Per_Capita_2019'}, inplace=True)

#print a sample dataset
print(gdp_data.head(1))

#save the final dataset
gdp_data.to_csv('data/gdp_data.csv', index=False)

  Country  GDP_Per_Capita_2015  GDP_Per_Capita_2016  GDP_Per_Capita_2017  \
0   Aruba         28396.908423         28452.170615         29350.805019   

   GDP_Per_Capita_2018  GDP_Per_Capita_2019  
0         30253.279358                  NaN  


OSError: Cannot save file into a non-existent directory: 'preprocessed'

In [ ]:
#do the same as above for the gdp_per_capita_growth dataset

gdp_growth_data = pd.read_csv('data/dataset1/gdp_per_capita_growth.csv')

#keep only the columns we need and rename to the universal index: Country
gdp_growth_data = gdp_growth_data[['Country Name', '2015', '2016', '2017', '2018', '2019']]
gdp_growth_data.rename(columns={'Country Name': 'Country'}, inplace=True)

#rename all yearly columns to gdp_per_capita_growth_x
gdp_growth_data.rename(columns={'2015': 'GDP_Per_Capita_Growth_2015'}, inplace=True)
gdp_growth_data.rename(columns={'2016': 'GDP_Per_Capita_Growth_2016'}, inplace=True)
gdp_growth_data.rename(columns={'2017': 'GDP_Per_Capita_Growth_2017'}, inplace=True)
gdp_growth_data.rename(columns={'2018': 'GDP_Per_Capita_Growth_2018'}, inplace=True)
gdp_growth_data.rename(columns={'2019': 'GDP_Per_Capita_Growth_2019'}, inplace=True)

#print a sample dataset
print(gdp_growth_data.head(1))

#save the final dataset
gdp_growth_data.to_csv('data/gdp_growth_data.csv', index=False)

  Country  GDP_Per_Capita_Growth_2015  GDP_Per_Capita_Growth_2016  \
0   Aruba                    5.129657                    1.587869   

   GDP_Per_Capita_Growth_2017  GDP_Per_Capita_Growth_2018  \
0                    1.519821                         NaN   

   GDP_Per_Capita_Growth_2019  
0                         NaN  
